# Original Data + FX Forecast Benchmark

This notebook uses the controlled FX-enhanced dataset at `data/merged/od_fx_merged.csv`. It mirrors the original data benchmark workflow while extending the input panel only with selected FX predictors from `data/processed/bank-of-botswana-exchange-rates_processed.csv`.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "src").exists() and (candidate / "data").exists() and (candidate / "notebooks").exists():
        repo_root = candidate.resolve()
        break
if repo_root is None:
    repo_root = cwd.resolve()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import os
os.chdir(repo_root)

repo_root, (repo_root / "data" / "merged" / "od_fx_merged.csv")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from lightgbm import LGBMRegressor

try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None

DATA_PATH = repo_root / "data" / "merged" / "od_fx_merged.csv"
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

target = "food_price_inflation"
feature_cols = [col for col in df.columns if col not in {"Date", target}]

X = df[feature_cols].copy()
y = df[target].astype(float).copy()

date_start = df["Date"].min()
date_end = df["Date"].max()
print(f"Dataset shape: {df.shape}")
print(f"Feature count: {len(feature_cols)}")
print(f"Date range: {date_start} -> {date_end}")


In [ ]:
def make_model(name):
    if name == "random_forest":
        return RandomForestRegressor(random_state=7, n_estimators=200, max_depth=None, min_samples_leaf=2, n_jobs=-1)
    if name == "lightgbm":
        return LGBMRegressor(random_state=7, n_estimators=250, learning_rate=0.05, num_leaves=15, subsample=0.9, colsample_bytree=0.9, objective="regression_l2")
    if name == "xgboost" and XGBRegressor is not None:
        return XGBRegressor(random_state=7, n_estimators=250, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, objective="reg:squarederror")
    raise ValueError(f"{name} is unavailable in this environment")

origin_years = [2021, 2022, 2023]
results = []

for year in origin_years:
    train_mask = df["Date"] < pd.Timestamp(f"{year}-01-01")
    test_mask = df["Date"].between(pd.Timestamp(f"{year}-01-01"), pd.Timestamp(f"{year}-12-01"))

    X_train = X.loc[train_mask].fillna(X.median(numeric_only=True))
    y_train = y.loc[train_mask].fillna(y.median())
    X_test = X.loc[test_mask].fillna(X.median(numeric_only=True))
    y_test = y.loc[test_mask].fillna(y.median())

    for model_name in ["random_forest", "lightgbm"]:
        model = make_model(model_name)
        model.fit(X_train, y_train)
        prediction = model.predict(X_test)
        results.append({
            "origin_year": year,
            "model": model_name,
            "rmse": float(np.sqrt(mean_squared_error(y_test, prediction))),
            "mae": float(mean_absolute_error(y_test, prediction)),
            "r2": float(r2_score(y_test, prediction)) if len(y_test) > 1 else np.nan,
        })

results_df = pd.DataFrame(results)
display(results_df)


In [ ]:
summary = results_df.groupby("model").agg({
    "mae": "mean",
    "rmse": "mean",
    "r2": "mean",
}).sort_values("rmse")
summary


In [ ]:
import matplotlib.pyplot as plt

comparison_frames = []
for year in origin_years:
    train_mask = df["Date"] < pd.Timestamp(f"{year}-01-01")
    test_mask = df["Date"].between(pd.Timestamp(f"{year}-01-01"), pd.Timestamp(f"{year}-12-01"))

    X_train = X.loc[train_mask].fillna(X.median(numeric_only=True))
    y_train = y.loc[train_mask].fillna(y.median())
    X_test = X.loc[test_mask].fillna(X.median(numeric_only=True))
    y_test = y.loc[test_mask].fillna(y.median())

    for model_name in ["random_forest", "lightgbm"]:
        model = make_model(model_name)
        model.fit(X_train, y_train)
        prediction = model.predict(X_test)
        comparison_frames.append(
            pd.DataFrame({
                "Date": df.loc[test_mask, "Date"].reset_index(drop=True),
                "origin_year": year,
                "model": model_name,
                "actual": y_test.reset_index(drop=True),
                "predicted": prediction,
            })
        )

comparison_df = pd.concat(comparison_frames, ignore_index=True)

for year in origin_years:
    fig, ax = plt.subplots(figsize=(10, 5))
    subset = comparison_df[comparison_df["origin_year"] == year]
    for model_name in ["random_forest", "lightgbm"]:
        model_subset = subset[subset["model"] == model_name]
        ax.plot(model_subset["Date"], model_subset["actual"], label="actual", color="black", linewidth=2)
        ax.plot(model_subset["Date"], model_subset["predicted"], label=f"predicted ({model_name})", linestyle="--", alpha=0.8)

    ax.set_title(f"Actual vs Predicted for {year} Origin")
    ax.set_xlabel("Date")
    ax.set_ylabel(target)
    ax.legend()
    ax.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## Actual vs Predicted Comparison

The following figures show 2021, 2022, and 2023 test-window actual vs predicted values for both benchmark models.


## Notes

- This notebook uses the controlled FX dataset at `data/merged/od_fx_merged.csv`.
- It is intentionally separate from `notebooks/original_data_forecast.ipynb` and the Stage C optimization workflow.
- The feature panel here includes only original data plus selected FX predictors; it does not add additional external sources.
